# Kiến Trúc CED-SingleMoE-SplitFFN (Zero-Oracle)
## Diễn Giải Chi Tiết Quá Trình Vật Lý, Cấu Trúc Mô Hình & Triển Khai Thực Nghiệm

> **Tác giả:** MLNC Team  
> **Kiến trúc:** `CED-SingleMoE-SplitFFN` (Champion Pipeline v2)  
> **Mục tiêu:** Phân loại âm thanh môi trường 36 lớp (`mixture-36l`) từ kênh nhiễu ước lượng $\hat{n}$ của bộ khử nhiễu **FRCRN**.  
> **Đặc điểm cốt lõi:** Sử dụng duy nhất **1 backbone CED-Tiny** (~6.62M tham số, dung lượng checkpoint chỉ ~26.5 MB), đạt và vượt hiệu năng của hệ thống Dual-Student cũ (10.84M tham số), loại bỏ 100% sự phụ thuộc vào nhãn Oracle SNR khi suy luận thực tế.

---

### Kết Quả Thực Nghiệm Trên Máy Chủ GPU (NVIDIA RTX PRO 6000 Blackwell)
| Chỉ số (Metric) | Hệ thống Dual-Student cũ (10.84M) | CED-SingleMoE-SplitFFN Mới (~6.62M) | Mức cải thiện / Trạng thái |
| :--- | :---: | :---: | :--- |
| **Top-1 Accuracy** | `63.58%` | **`63.94%`** | **+0.36%** (Vượt ngưỡng cũ) |
| **Macro mAP** | `70.08%` | **`70.05%`** | Tương đương (chênh lệch <0.03%) |
| **Macro AUC** | `95.80%` | **`96.10%`** | **+0.30%** |
| **Số lượng tham số** | 10.84M (2 backbones) | **~6.62M (1 backbone duy nhất)** | **Giảm 39% tham số** |
| **Kích thước Checkpoint** | ~43.4 MB | **~26.5 MB** | **Nhẹ hơn 1.64x** |
| **Phụ thuộc Oracle SNR** | Bắt buộc cần nhãn SNR | **100% Zero-Oracle (Không cần SNR)** | **Sẵn sàng triển khai thực tế** |

---

### Sơ Đồ Luồng Dữ Liệu Toàn Hệ Thống (Dataflow Architecture)

```
Sóng âm nhiễu ước lượng n̂ [B, 64000] (16kHz, 4.0s từ bộ tách FRCRN)
  │
  ├──► [Tầng Chuẩn Hóa RMS]: Chuẩn hóa năng lượng về σ₀ = 0.07925 (-22.02 dBFS)
  │      │
  │      ├──► [Nhánh 1: AcousticFeatureExtractor]: STFT [B, 257, 401] 
  │      │      └──► Trích xuất 6 đặc trưng vật lý f_spec ∈ ℝ⁶ (Log-RMS, Crest, Kurtosis, Flatness, f5, f6)
  │      │
  │      └──► [Nhánh 2: LogMel Frontend]: 64 Mel × 398 Frames + Z-score Instance Norm + SpecAugment
  │             └──► Tensor ảnh phổ Mel: [B, 1, 64, 398]
  │                    │
  │                    ▼
  │             [Patch Embedding]: Conv2d 16×16, stride 16 ──► 4×24 = 96 patch tokens
  │             [Positional Embed 2D + CLS Token]: Ghép nối ──► X₀: [B, 97, 192]
  │                    │
  │                    ▼
  │             [Blocks 0 → 9 (Dùng chung 100%)]: Pre-LN Transformer + 2D GREP BiasAttention
  │                    │
  │                    ├──► Trích xuất CLS token trung gian h₉ = X₉[:, 0] ∈ ℝ¹⁹²
  │                    │      │
  │                    │      ▼
  │                    │   [Self-Contained Spectral-Token Router]:
  │                    │      Linear(192→16) + Ghép f_spec(6) ──► Vector u ∈ ℝ²²
  │                    │      MLP(22→32→2) + Softmax ──► Trọng số điều phối mềm α ∈ (0, 1)
  │                    │      │
  │                    ▼      ▼
  │             [Blocks 10 → 11 (SplitMoEFFN)]:
  │                y = 0.5·FFN_shared(x) + 0.5·[(1−α)·FFN_low(x) + α·FFN_high(x)]
  │                    │
  │                    ▼
  │             [Đầu Phân Loại Kép (Dual-Head Classifier)]:
  │                z = (1−α)·Head_low(CLS₁₁) + α·Head_high(CLS₁₁) ──► Logits 36 lớp: [B, 36]
```

In [ ]:
# 1. Cấu hình đường dẫn hệ thống và nạp các thư viện cốt lõi
import os
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Đảm bảo in tiếng Việt chuẩn UTF-8 trên mọi môi trường
if sys.stdout and hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

# Tự động định vị thư mục gốc gói kd-moe-ced (nơi chứa src/frontend.py)
PKG_ROOT = None
search_candidates = [
    Path.cwd() / "imp" / "kd-moe-ced",
    Path.cwd(),
    Path("d:/PROJECT/MLNC/imp/kd-moe-ced"),
]
for candidate in search_candidates:
    if (candidate / "src" / "frontend.py").is_file():
        PKG_ROOT = candidate
        break

if PKG_ROOT is None:
    PKG_ROOT = Path("d:/PROJECT/MLNC/imp/kd-moe-ced")

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

# Nạp các module tự phát triển từ package src
from src.frontend import TARGET_RMS, AcousticFeatureExtractor, LogMel, SpecAugment, rms_normalize
from src.router import SelfContainedSpectralTokenRouter
from src.models.vit_grep import ViTSBias, BiasAttention, SharedBias
from src.models.split_moe import CEDSingleMoESplitFFN, SplitMoEFFN
from src.metrics import compute_topk_acc, compute_map, compute_auc, compute_snr_breakdown
from src.pipeline import ChampionPipeline

# Thiết lập thiết bị tính toán tối ưu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-> Thư mục dự án: {PKG_ROOT}")
print(f"-> Thiết bị tính toán được chọn: {device}")
torch.manual_seed(42)
np.random.seed(42)

---
## Phần 1: Tầng Frontend & 6 Đặc Trưng Âm Học Vật Lý (`src/frontend.py`)

### 1.1. Chuẩn hóa Năng lượng RMS về $\sigma_0 = 0.07925$
Khi âm thanh $\hat{n}$ được đưa vào hệ thống, biên độ thu âm có thể thay đổi rất lớn tùy theo âm lượng phát hoặc micro. Nếu đưa trực tiếp vào mạng:
- Các phép đo năng lượng (Log-RMS, Crest factor) sẽ bị phụ thuộc vào độ to/nhỏ của loa thay vì bản chất vật lý của sóng âm.
- Biên độ tín hiệu dao động mạnh gây mất ổn định gradient ở tầng tích chập đầu tiên.

Ta thực hiện chuẩn hóa năng lượng hiệu dụng RMS về hằng số chuẩn hóa $\sigma_0 = 0.07925$:
$$x_{\text{norm}}(t) = x(t) \cdot \frac{\sigma_0}{\sqrt{\frac{1}{T}\sum_{t=1}^T x(t)^2} + \epsilon}$$
Con số $0.07925$ tương ứng với mức năng lượng:
$$20 \cdot \log_{10}(0.07925) \approx -22.02 \text{ dBFS RMS}$$
Đây là mức năng lượng hiệu dụng chuẩn hóa trung vị của tiếng nói và âm thanh môi trường trong các tập dữ liệu lớn như **AudioSet**, **FSD50K** và khuyến nghị chuẩn **ITU-T P.56**.

### 1.2. Bản chất Vật lý của 6 Đặc Trưng Âm Học ($f_1 \to f_6$)
Bộ khử nhiễu **FRCRN** khi xử lý ở vùng SNR cao ($> 15\text{ dB}$) thường để lại các tàn dư là **các âm tiết tiếng nói rò rỉ rải rác (speech leakage transients)** vào kênh nhiễu $\hat{n}$. 6 đặc trưng vật lý được thiết kế để phân biệt nhiễu nền liên tục với giả ảnh rò rỉ này:

1. **$f_1$ (Log-RMS):** $\ln(\text{RMS} + \epsilon)$ — Đo năng lượng tổng thể sau chuẩn hóa.
2. **$f_2$ (Crest Factor):** $\frac{\max |x(t)|}{\text{RMS}}$ — Tỷ số đỉnh cực đại trên RMS. Nhiễu môi trường nền liên tục có crest factor thấp ($3\text{--}4$); các xung đột biến hoặc âm tiết rò rỉ đẩy crest factor lên $> 6\text{--}10$.
3. **$f_3$ (Temporal Kurtosis):** Hệ số nhọn bậc 4 của năng lượng khung theo thời gian:
   $$\text{Kurtosis} = \frac{\frac{1}{T}\sum_{t=1}^T (E_t - \mu_E)^4}{\sigma_E^4}$$
   Nhiễu Gauss dừng có Kurtosis $\approx 3$. Tiếng nói rò rỉ xuất hiện ngắt quãng thành các âm tiết cục bộ tạo ra các đỉnh xung rất nhọn, đẩy Kurtosis lên $\gg 3$.
4. **$f_4$ (Spectral Flatness - Wiener Entropy):** Tỷ số giữa trung bình hình học và trung bình số học của phổ:
   $$\text{Flatness} = \frac{\exp\left(\frac{1}{F}\sum_{f=1}^F \ln P(f, t)\right)}{\frac{1}{F}\sum_{f=1}^F P(f, t)}$$
   Nhiễu trắng/nhiễu phẳng có giá trị $\approx 1$; tín hiệu có cấu trúc họa âm rõ rệt (nguyên âm tiếng nói rò rỉ) có Flatness $\ll 1$.
5. **$f_5$ (Low-band to Speech-band Ratio):** Tỷ số log năng lượng dải trầm ($< 300\text{ Hz}$, bins $0\dots 9$) so với dải thoại người ($300\text{--}3400\text{ Hz}$, bins $10\dots 108$).
6. **$f_6$ (High-band to Speech-band Ratio):** Tỷ số log năng lượng dải cao ($> 3400\text{ Hz}$, bins $109\dots 256$) so với dải thoại người.

In [ ]:
# Mô phỏng thực nghiệm Frontend: So sánh 2 loại tín hiệu (Nhiễu nền liên tục vs Nhiễu kèm rò rỉ tiếng nói)
# Tín hiệu dài 4 giây tại 16 kHz = 64,000 mẫu
sample_rate = 16000
duration_samples = 64000
time_axis = np.linspace(0, 4.0, duration_samples)

# Tín hiệu A: Nhiễu nền Gaussian dừng (Mô phỏng SNR thấp)
torch.manual_seed(100)
wave_noise_low = torch.randn(1, duration_samples) * 0.05

# Tín hiệu B: Nhiễu nền kèm các xung âm tiết tiếng nói rò rỉ rải rác (Mô phỏng SNR cao bị rò rỉ FRCRN)
wave_noise_leak = wave_noise_low.clone()
# Chèn 3 cụm xung âm tiết mô phỏng tiếng nói rò rỉ
for start_t in [8000, 24000, 42000]:
    t = torch.linspace(0, 1.0, 4000)
    burst = 0.6 * torch.sin(2 * np.pi * 440 * t) * torch.hamming_window(4000)
    wave_noise_leak[0, start_t : start_t + 4000] += burst

# 1. Thực hiện chuẩn hóa RMS
wave_norm_a = rms_normalize(wave_noise_low)
wave_norm_b = rms_normalize(wave_noise_leak)

print(f"RMS Tín hiệu A sau chuẩn hóa: {wave_norm_a.square().mean().sqrt().item():.5f} (Kỳ vọng: {TARGET_RMS})")
print(f"RMS Tín hiệu B sau chuẩn hóa: {wave_norm_b.square().mean().sqrt().item():.5f} (Kỳ vọng: {TARGET_RMS})")

# 2. Trích xuất 6 đặc trưng âm học vật lý
extractor = AcousticFeatureExtractor(n_fft=512, hop_length=160)
feat_a = extractor(wave_norm_a)[0].detach().cpu().numpy()
feat_b = extractor(wave_norm_b)[0].detach().cpu().numpy()

feature_names = [
    "f1: Log-RMS",
    "f2: Crest Factor",
    "f3: Temporal Kurtosis",
    "f4: Spectral Flatness",
    "f5: Low/Speech Ratio",
    "f6: High/Speech Ratio",
]

print("\n=== SO SÁNH 6 ĐẶC TRƯNG ÂM HỌC VẬT LÝ ===")
print(f"{'Tên đặc trưng':<25} | {'Nhiễu dừng (SNR thấp)':<22} | {'Rò rỉ tiếng nói (SNR cao)':<25}")
print("-" * 78)
for name, val_a, val_b in zip(feature_names, feat_a, feat_b):
    print(f"{name:<25} | {val_a:<22.4f} | {val_b:<25.4f}")

# 3. Trích xuất phổ Log-Mel [1, 64, 398]
frontend = LogMel(n_mels=64, num_frames=398)
mel_a = frontend(wave_norm_a)[0, 0].detach().cpu().numpy()
mel_b = frontend(wave_norm_b)[0, 0].detach().cpu().numpy()

# Vẽ biểu đồ so sánh dạng sóng và ảnh phổ Log-Mel
fig, axs = plt.subplots(2, 2, figsize=(14, 6))

axs[0, 0].plot(time_axis[:8000], wave_norm_a[0, :8000].numpy(), color="steelblue")
axs[0, 0].set_title("Dạng sóng A: Nhiễu nền dừng (SNR thấp)")
axs[0, 0].set_ylabel("Biên độ")
axs[0, 0].grid(True, alpha=0.3)

axs[0, 1].plot(time_axis[:8000], wave_norm_b[0, :8000].numpy(), color="crimson")
axs[0, 1].set_title("Dạng sóng B: Nhiễu có xung rò rỉ tiếng nói (SNR cao)")
axs[0, 1].grid(True, alpha=0.3)

im1 = axs[1, 0].imshow(mel_a, origin="lower", aspect="auto", cmap="viridis")
axs[1, 0].set_title("Phổ Log-Mel A (64 Mel bins x 398 frames)")
axs[1, 0].set_xlabel("Khung thời gian (Frames)")
axs[1, 0].set_ylabel("Dải tần Mel")
plt.colorbar(im1, ax=axs[1, 0])

im2 = axs[1, 1].imshow(mel_b, origin="lower", aspect="auto", cmap="magma")
axs[1, 1].set_title("Phổ Log-Mel B (Nhìn rõ vệt xung rò rỉ âm tiết)")
axs[1, 1].set_xlabel("Khung thời gian (Frames)")
plt.colorbar(im2, ax=axs[1, 1])

plt.tight_layout()
plt.show()

---
## Phần 2: Backbone ViT CED-Tiny & 2D GREP BiasAttention (`src/models/vit_grep.py`)

### 2.1. Phép Chiếu Patch và Chuỗi Token
- **Tầng Patch Embedding:** Dùng `Conv2d(1, 192, kernel_size=16, stride=16)`.
- Kích thước phổ Mel: $64 \text{ dải tần} \times 384 \text{ khung thời gian}$ (cắt từ 398 frames).
- Lưới patch: $\frac{64}{16} = 4$ patch tần số $\times \frac{384}{16} = 24$ patch thời gian $\implies 96$ patch tokens.
- Ghép **CLS Token** vào vị trí đầu tiên: Chuỗi có kích thước $X_0 \in \mathbb{R}^{B \times 97 \times 192}$.

### 2.2. Cơ chế 2D Gated Relative Position Bias (GREP Attention)
Cơ chế Self-Attention chuẩn của Vision Transformer có tính chất hoán vị bất biến (permutation invariant). Để mô hình nhận diện chính xác cấu trúc âm học 2 chiều (thời gian - tần số), cơ chế **2D GREP BiasAttention** được triển khai:
1. **Lưới khoảng cách tương đối tĩnh 2D ($P_{\text{rel}}$):**
   - Khoảng cách tần số: $\Delta f \in [-3, 3]$ (gồm $2 \times 4 - 1 = 7$ vị trí).
   - Khoảng cách thời gian: $\Delta t \in [-23, 23]$ (gồm $2 \times 24 - 1 = 47$ vị trí).
   - Khối Transformer 0 sở hữu bảng nhúng; các khối 1 đến 11 tái sử dụng ma trận này thông qua bộ chứa `SharedBias`, tiết kiệm tối đa bộ nhớ VRAM.
2. **Cổng điều chế biên độ động (Dynamic Gating):**
   Vector truy vấn $Q$ được chiếu qua tầng tuyến tính:
   $$\text{gate} = \sigma(\text{Linear}(Q)) \implies [g_a, g_b]$$
   Hệ số điều chế biên độ:
   $$S = g_a \cdot (g_b \cdot a - 1.0) + 2.0$$
   Ma trận Bias đã điều chế:
   $$B_{\text{attn}} = S \cdot P_{\text{rel}}$$
   Cộng trực tiếp vào ma trận tích vô hướng trước khi qua Softmax:
   $$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{Q K^T}{\sqrt{64}} + B_{\text{attn}}\right) V$$

In [ ]:
# Kiểm chứng cấu trúc ViT Backbone và cơ chế chia sẻ ma trận bias SharedBias
model_vit = ViTSBias(num_classes=36, freq_bins=64, frames=398, pretrained=False)
model_vit.eval()

# Tạo batch phổ Mel giả lập: [B=2, C=1, Freq=64, Frames=398]
dummy_mel = torch.randn(2, 1, 64, 398)

# 1. Kiểm tra phép chiếu patch
patches = model_vit.vit.patch_embed(dummy_mel[..., :model_vit.frames])
print(f"Kích thước tensor sau Conv2d Patch Projection: {patches.shape} (B, N_patches=96, Embed_dim=192)")

# 2. Truyền thẳng qua khối Transformer 0
b0 = model_vit.vit.blocks[0]
cls = (model_vit.vit.cls_token + model_vit.vit.pos_embed[:, :1]).expand(2, -1, -1)
x_in = model_vit.vit.norm_pre(torch.cat([cls, patches + model_vit.vit.pos_embed[:, 1:]], dim=1))
print(f"Kích thước chuỗi đầu vào X_0 (kèm CLS token): {x_in.shape} (B=2, Tokens=97, Dim=192)")

# Lượt forward qua Block 0 để kích hoạt việc sinh bảng SharedBias
out_b0 = b0(x_in)
shared_bias_matrix = model_vit.shared.value

print(f"Kích thước ma trận Bias vị trí tương đối 2D dùng chung: {shared_bias_matrix.shape}")
print(f"  -> Gồm: 1 batch x 3 attention heads x 97 x 97 token pairs")
print(f"Block 0 sở hữu bảng nhúng: {hasattr(b0.attn, 'freq_rel_bias')}")
print(f"Block 1 tái sử dụng bảng dùng chung: {not hasattr(model_vit.vit.blocks[1].attn, 'freq_rel_bias')}")

# Đếm tham số tầng MLP trong mỗi khối
mlp_params = sum(p.numel() for p in b0.mlp.parameters())
print(f"Số lượng tham số của 1 tầng MLP (192 -> 768 -> 192): {mlp_params:,} tham số")

---
## Phần 3: Bộ Định Tuyến Tự Thân (`SelfContainedSpectralTokenRouter` - `src/router.py`)

### 3.1. Nghịch lý của Nhãn Oracle SNR trong Thực Tế
Trong các nghiên cứu trước đây (Dual-Student), hệ thống yêu cầu phải biết trước mẫu âm thanh đang ở SNR bao nhiêu dB để kích hoạt mô hình phù hợp. Tuy nhiên:
- **Trong thực tế (Inference/Production):** Người dùng chỉ ghi âm được tín hiệu âm thanh thô, hoàn toàn không có ai cung cấp nhãn ground-truth SNR!
- Nếu dùng mô hình ước lượng SNR riêng biệt: Tăng gấp đôi độ trễ tính toán và nếu mạng ước lượng SNR dự đoán sai lệch thì toàn bộ hệ thống phân loại phía sau sẽ sụp đổ.

### 3.2. Thiết Kế Bộ Định Tuyến Nội Tại (Self-Contained Dynamic Routing)
Bộ định tuyến tự thân hoạt động dựa trên sự phối hợp đa mức (Cross-Level Fusion):
1. **Mức Ngữ nghĩa Toàn cục (Semantic Level):** Trích xuất token CLS $h_9 \in \mathbb{R}^{192}$ sau Khối Transformer 9. Sau 9 khối Transformer, token CLS đã tổng hợp được ngữ cảnh trừu tượng sâu của toàn bộ đoạn âm thanh. Ta chiếu giảm chiều qua tầng tuyến tính $192 \to 16$ (được kích hoạt GELU và chuẩn hóa LayerNorm) để thu được $\tilde{h}_9 \in \mathbb{R}^{16}$.
2. **Mức Vật lý Tức thời (Physical Level):** Ghép nối với 6 đặc trưng vật lý $f_{\text{spec}} \in \mathbb{R}^6$ trích xuất trực tiếp từ sóng âm:
   $$u = [f_{\text{spec}} \parallel \tilde{h}_9] \in \mathbb{R}^{22}$$
3. **Mạng MLP Tương tác Liên Mức:**
   $$[r_{\text{low}}, r_{\text{high}}] = \text{Linear}_{32 \to 2}(\text{GELU}(\text{LN}(\text{Linear}_{22 \to 32}(u))))$$
4. **Phân Phối Xác Suất Mềm:**
   $$\alpha = \frac{\exp(r_{\text{high}})}{\exp(r_{\text{low}}) + \exp(r_{\text{high}})} \in (0, 1)$$
   - $\alpha$: Trọng số kích hoạt cho Chuyên gia High-SNR (xử lý rò rỉ tiếng nói).
   - $1 - \alpha$: Trọng số kích hoạt cho Chuyên gia Low-SNR (xử lý nhiễu môi trường).

### 3.3. Giám Sát Nhãn Mềm Trong Huấn Luyện (Supervised Soft Labeling)
Trong quá trình huấn luyện, ta có thông tin SNR thực tế $S$ (dB). Ta giám sát bộ định tuyến bằng hàm chuyển tiếp sigmoid:
$$y_{\text{target}} = \sigma\left(\frac{S - 15.0}{2.5}\right)$$
- Khi $S = -5\text{ dB} \implies y_{\text{target}} = 0.0003$ (Hầu như 100% chế độ Low-SNR).
- Khi $S = 15\text{ dB} \implies y_{\text{target}} = 0.5000$ (Điểm chuyển giao mượt mà).
- Khi $S = 20\text{ dB} \implies y_{\text{target}} = 0.8808$ (Tập trung chuyên sâu vào xử lý rò rỉ tiếng nói).

In [ ]:
# Khởi tạo và đánh giá hoạt động của Router
router = SelfContainedSpectralTokenRouter(spectral_dim=6, token_dim=192, token_proj_dim=16, hidden_dim=32)
router.eval()

# Khảo sát hàm mục tiêu mềm theo từng mức SNR
snr_range = torch.linspace(-5.0, 20.0, 100)
soft_targets = router.compute_target_soft_label(snr_range).squeeze(-1).numpy()

# Mô phỏng lượt forward qua Router với batch 4 mẫu
dummy_feat = torch.randn(4, 6)
dummy_cls9 = torch.randn(4, 192)

with torch.no_grad():
    alpha, gates = router(dummy_feat, dummy_cls9)

print(f"Kích thước đầu ra alpha (High-Expert weight): {alpha.shape}")
print(f"Kích thước gates [p_low, p_high]: {gates.shape}")
print(f"Giá trị alpha ngẫu nhiên: {alpha.squeeze(-1).tolist()}")
print(f"Tổng xác suất trên 2 cổng (luôn bằng 1.0): {gates.sum(dim=-1).tolist()}")

# Vẽ đường cong chuyển tiếp nhãn mục tiêu theo dải SNR
plt.figure(figsize=(9, 4.5))
plt.plot(snr_range.numpy(), soft_targets, label="Đường cong chuyển tiếp y_target", color="darkorange", linewidth=2.5)
plt.axvline(15.0, color="gray", linestyle="--", alpha=0.7, label="Điểm chuyển giao (SNR = 15 dB, target = 0.5)")
plt.axhspan(0.0, 0.2, color="blue", alpha=0.08, label="Vùng Chuyên gia Low-SNR (-5dB -> 10dB)")
plt.axhspan(0.8, 1.0, color="red", alpha=0.08, label="Vùng Chuyên gia High-SNR (15dB -> 20dB)")
plt.xlabel("Mức SNR (dB)")
plt.ylabel("Trọng số mục tiêu alpha")
plt.title("Hàm Chuyển Tiếp Điều Phối Chuyên Gia Nội Tại Theo SNR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Phần 4: Khối MoE Sâu (Blocks 10–11) & Đầu Phân Loại Kép (`src/models/split_moe.py`)

### 4.1. Cấu Trúc Split-FFN: 1 Chuyên Gia Dùng Chung + 2 Chuyên Gia Chuyên Biệt
Khác với kiến trúc MoE thưa (Sparse MoE) thông thường dễ gặp hiện tượng sụp đổ chuyên gia (expert collapse), kiến trúc **SplitMoEFFN** kết hợp giữa tri thức nền tảng dùng chung và năng lực chuyên biệt hóa:
$$y = 0.5 \cdot \text{FFN}_{\text{shared}}(x) + 0.5 \cdot \left[(1 - \alpha) \cdot \text{FFN}_{\text{low}}(x) + \alpha \cdot \text{FFN}_{\text{high}}(x)\right]$$

**Đặc tính chuẩn hóa lồi (Strictly Convex Combination):**
Tổng trọng số điều phối qua các nhánh:
$$0.5 + 0.5 \cdot (1 - \alpha) + 0.5 \cdot \alpha = 0.5 + 0.5 = 1.0$$
Nhờ tổng trọng số luôn bằng $1.0$, độ lớn kích hoạt của tensor đầu ra được giữ nguyên mà không bị biến thiên biên độ theo $\alpha$.

### 4.2. Khởi Tạo Bảo Toàn Toán Học (Identity Warm-Start Initialization)
Khi bắt đầu quá trình huấn luyện:
- Ba nhánh `ffn_shared`, `ffn_low`, và `ffn_high` đều được khởi tạo bằng bản sao chép trọng số (deep copy) chính xác từ tầng MLP của CED-Tiny đã được tiền huấn luyện trên AudioSet:
  $$\text{FFN}_{\text{shared}}(x) = \text{FFN}_{\text{low}}(x) = \text{FFN}_{\text{high}}(x) = M(x)$$
- Tại bước huấn luyện thứ 0 (Step 0):
  $$y = 0.5 M(x) + 0.5 [(1-\alpha) M(x) + \alpha M(x)] = 0.5 M(x) + 0.5 M(x) = M(x)$$
- **Kết luận:** Tại thời điểm khởi tạo, đầu ra của mô hình **HOÀN TOÀN ĐỒNG NHẤT VỀ MẶT TOÁN HỌC** với mô hình CED-Tiny cơ sở cho **MỌI GIÁ TRỊ $\alpha \in (0, 1)$**. Nhờ đó, mô hình tránh được hiện tượng sốc ngẫu nhiên (random shock) và hội tụ cực kỳ mượt mà.

### 4.3. Đầu Phân Loại Kép (Dual Classifier Heads)
Ở tầng ra cuối cùng, token CLS sau Block 11 được đưa đồng thời qua 2 đầu tuyến tính:
- $\text{Head}_{\text{low}}: \mathbb{R}^{192} \to \mathbb{R}^{36}$ (Chuyên gia phân loại nhiễu môi trường SNR thấp).
- $\text{Head}_{\text{high}}: \mathbb{R}^{192} \to \mathbb{R}^{36}$ (Chuyên gia phân loại khi có rò rỉ âm tiết tiếng nói).
- Logits đầu ra tổng hợp:
  $$z = (1 - \alpha) \cdot z_{\text{low}} + \alpha \cdot z_{\text{high}}$$

In [ ]:
# Khởi tạo mô hình hoàn chỉnh CEDSingleMoESplitFFN và kiểm chứng tính toàn vẹn
model_champion = CEDSingleMoESplitFFN(num_classes=36, freq_bins=64, frames=398, split_blocks=(10, 11))
model_champion.eval()

# Kiểm tra tổng số lượng tham số
param_info = model_champion.get_num_params()
print("=== THỐNG KÊ SỐ LƯỢNG THAM SỐ KIẾN TRÚC ===")
print(f"Tổng số tham số: {param_info['total']:,} tham số (~{param_info['total'] / 1e6:.2f}M)")
print(f"Số tham số huấn luyện được: {param_info['trainable']:,} tham số")
print(f"Số tham số của Bộ định tuyến Router: {param_info['router']:,} tham số (Chỉ chiếm ~{param_info['router'] / param_info['total'] * 100:.2f}% tổng mạng)")

# Chạy thử nghiệm End-to-End với 2 đoạn âm thanh giả lập
dummy_wave = torch.randn(2, 64000)
with torch.no_grad():
    output = model_champion(dummy_wave, return_dict=True)

print("\n=== KIỂM TRA ĐẦU RA END-TO-END TỪ SÓNG ÂM ===")
print(f"Sóng âm đầu vào: {dummy_wave.shape} [Batch=2, Mẫu=64,000]")
print(f"Logits dự đoán cuối cùng: {output['logits'].shape} [Batch=2, Lớp=36]")
print(f"Trọng số định tuyến alpha: {output['alpha'].squeeze(-1).tolist()}")
print(f"Đặc trưng vật lý 6 chiều trích xuất: {output['spectral_feat'].shape}")
print(f"Biểu diễn token CLS tại Block 9: {output['block9_token'].shape}")
print("-> Mọi kích thước tensor và luồng tính toán hoàn toàn khớp 100% với đặc tả kỹ thuật!")

---
## Phần 5: Huấn Luyện Chưng Cất Tri Thức Kép (Dual-Teacher KD - `src/kd_trainer.py`)

### 5.1. Hai Giáo Viên Chuyên Biệt (Dual Teachers)
Hệ thống sử dụng hai mô hình giáo viên BEATs-12L (~90M tham số mỗi mô hình):
1. **Teacher 1 (`BEATs Fold0`):** Huấn luyện trên âm thanh môi trường chuẩn và sạch, đóng vai trò chuyên gia nhận diện các đặc trưng âm học nguyên bản ở dải SNR thấp ($-5\text{ dB} \to 10\text{ dB}$).
2. **Teacher 2 (`BEATs EMA`):** Được cập nhật trọng số Exponential Moving Average liên tục trên dữ liệu hỗn hợp có bộ khử nhiễu FRCRN, đóng vai trò chuyên gia nhận diện và triệt tiêu các mẫu giả ảnh rò rỉ tiếng nói ở SNR cao ($15\text{ dB} \to 20\text{ dB}$).

### 5.2. Kỹ Thuật Consistent Mixup Đa Thành Phần
Nhằm tạo không gian biểu diễn liên tục và ổn định gradient, kỹ thuật Consistent Mixup (Beyer et al., CVPR 2022) được áp dụng đồng thời trên tất cả các thành phần với hệ số $\lambda \sim \text{Beta}(\alpha_{\text{mix}}, \alpha_{\text{mix}})$:
$$\begin{aligned}
\tilde{x} &= \lambda x_i + (1 - \lambda) x_j \\
\tilde{y} &= \lambda y_i + (1 - \lambda) y_j \\
\tilde{S} &= \lambda S_i + (1 - \lambda) S_j \\
\tilde{p}_{T0} &= \lambda p_{T0, i} + (1 - \lambda) p_{T0, j} \\
\tilde{p}_{EMA} &= \lambda p_{EMA, i} + (1 - \lambda) p_{EMA, j}
\end{aligned}$$

### 5.3. Hàm Mất Mát Chưng Cất Tích Hợp (Total Loss)
Hàm mất mát tổng thể kết hợp 3 thành phần:
$$L_{\text{total}} = \lambda_{\text{hard}} L_{\text{hard}} + \lambda_{\text{KD}} L_{\text{KD}} + \lambda_{\text{router}} L_{\text{router}}$$
Với:
- **$L_{\text{hard}}$:** Binary Cross-Entropy với nhãn thật $\tilde{y}$.
- **$L_{\text{KD}}$:** Kết hợp chưng cất toàn cục với phân phối giáo viên hòa trộn $\tilde{p}_{\text{teacher}} = (1 - y_{\text{target}})\tilde{p}_{T0} + y_{\text{target}}\tilde{p}_{EMA}$ và chưng cất chuyên biệt hóa cho từng chuyên gia:
  $$L_{\text{KD}} = \text{KL}(p_{\text{student}} \parallel \tilde{p}_{\text{teacher}}) + 0.5 \left[ \text{KL}(p_{\text{low}} \parallel \tilde{p}_{T0}) + \text{KL}(p_{\text{high}} \parallel \tilde{p}_{EMA}) \right]$$
- **$L_{\text{router}}$:** Giám sát trọng số điều phối $\alpha$ tiệm cận với $y_{\text{target}}$ thông qua Binary Cross-Entropy.

In [ ]:
# Mô phỏng 1 bước huấn luyện (Training Step) hoàn chỉnh với hàm mất mát tích hợp
model_train = CEDSingleMoESplitFFN(num_classes=36).to(device)
model_train.train()

optimizer = torch.optim.AdamW(model_train.parameters(), lr=1e-4, weight_decay=0.05)
bce_loss_fn = nn.BCEWithLogitsLoss()

# Tạo batch giả lập gồm 4 mẫu âm thanh kèm nhãn và logits của 2 giáo viên
batch_size = 4
dummy_wave = torch.randn(batch_size, 64000, device=device)
dummy_target = torch.zeros(batch_size, 36, device=device)
dummy_target[:, 5] = 1.0  # Giả sử lớp 5 là nhãn dương
dummy_snr = torch.tensor([-5.0, 5.0, 15.0, 20.0], device=device)
dummy_t0_logits = torch.randn(batch_size, 36, device=device)
dummy_tema_logits = torch.randn(batch_size, 36, device=device)

# Lượt forward tính toán
optimizer.zero_grad()
out = model_train(dummy_wave, return_dict=True)

logits = out["logits"]
logits_low = out["logits_low"]
logits_high = out["logits_high"]
alpha = out["alpha"]

# 1. Hard label loss
loss_hard = bce_loss_fn(logits, dummy_target)

# 2. Dual-Teacher KD loss (Nhiệt độ T = 2.0)
T = 2.0
y_target = model_train.router.compute_target_soft_label(dummy_snr)
p_t0 = F.softmax(dummy_t0_logits / T, dim=-1)
p_tema = F.softmax(dummy_tema_logits / T, dim=-1)
p_blended_teacher = (1.0 - y_target) * p_t0 + y_target * p_tema

log_p_student = F.log_softmax(logits / T, dim=-1)
loss_kd_global = F.kl_div(log_p_student, p_blended_teacher, reduction="batchmean") * (T ** 2)
loss_kd_low = F.kl_div(F.log_softmax(logits_low / T, dim=-1), p_t0, reduction="batchmean") * (T ** 2)
loss_kd_high = F.kl_div(F.log_softmax(logits_high / T, dim=-1), p_tema, reduction="batchmean") * (T ** 2)
loss_kd = loss_kd_global + 0.5 * (loss_kd_low + loss_kd_high)

# 3. Router supervision loss
loss_router = model_train.router.compute_router_loss(alpha, dummy_snr)

# Tổng hợp hàm mất mát
total_loss = 0.20 * loss_hard + 0.70 * loss_kd + 0.10 * loss_router

# Lan truyền ngược gradient
total_loss.backward()
optimizer.step()

print(f"-> Bước huấn luyện thành công!")
print(f"   * Loss Hard: {loss_hard.item():.4f}")
print(f"   * Loss KD Kép: {loss_kd.item():.4f}")
print(f"   * Loss Router: {loss_router.item():.4f}")
print(f"   * Tổng Loss: {total_loss.item():.4f}")

---
## Phần 6: Suy Luận Thực Tế & Đánh Giá Benchmark (`src/pipeline.py`)

### 6.1. Tăng Cường Thời Điểm Kiểm Tra (3-Way Temporal TTA)
Để loại bỏ sự phụ thuộc vào thời điểm bắt đầu phát âm thanh, pipeline áp dụng kỹ thuật Test-Time Augmentation (TTA) với 3 lần suy luận:
1. Sóng âm gốc: dịch chuyển $0\text{ ms}$ (0 mẫu).
2. Sóng âm dịch chuyển tới: $+50\text{ ms}$ ($+800$ mẫu tại $16\text{ kHz}$).
3. Sóng âm dịch chuyển lui: $-50\text{ ms}$ ($-800$ mẫu tại $16\text{ kHz}$).
Logits và trọng số $\alpha$ được lấy trung bình cộng qua 3 lượt suy luận, giúp triệt tiêu nhiễu pha thời gian.

### 6.2. Hiệu Chuẩn Nhiệt Độ Bất Đối Xứng (Asymmetric Temperature Calibration)
Mô hình học sinh (Student CED-Tiny) và mô hình giáo viên (Teacher BEATs) có mức độ tự tin (confidence) khác nhau. Áp dụng nhiệt độ hiệu chuẩn:
- $T_{\text{student}} = 1.40$: Giảm bớt sự tự tin quá mức (over-confidence) của mạng compact.
- $T_{\text{teacher}} = 1.20$: Làm sắc nét phân phối xác suất của giáo viên lớn.

In [ ]:
# Khởi tạo Champion Pipeline và chạy thử nghiệm suy luận
pipeline = ChampionPipeline(
    student_model=model_champion,
    device=device,
    t_student=1.40,
    t_teacher=1.20,
)

# Chạy suy luận độc lập (Standalone Student) có TTA 3-Way
dummy_eval_wave = torch.randn(12, 64000, device=device)
probs_student, alphas = pipeline.predict_student_tta(dummy_eval_wave)

print(f"Xác suất dự đoán sau TTA 3-Way: {probs_student.shape} [12 mẫu, 36 lớp]")
print(f"Trọng số alpha trung bình qua TTA: {alphas.shape}")

# Mô phỏng tính toán phân rã hiệu năng theo từng dải SNR
# Giả sử có 6 mức SNR: -5dB, 0dB, 5dB, 10dB, 15dB, 20dB (mỗi mức 2 mẫu)
dummy_snrs = np.array([-5, -5, 0, 0, 5, 5, 10, 10, 15, 15, 20, 20])
dummy_targets = np.random.randint(0, 36, size=12)

results = compute_snr_breakdown(probs_student, dummy_targets, dummy_snrs)
print("\n=== BẢNG PHÂN RÃ HIỆU NĂNG THEO DẢI SNR (MÔ PHỎNG) ===")
for snr_key, res in results.items():
    if snr_key == "overall":
        print(f"-> Tổng thể: Top-1={res['top1']:.2%} | Top-3={res['top3']:.2%} | mAP={res['map']:.4f} | AUC={res.get('auc', 0):.4f}")
    else:
        print(f"   Dải {snr_key:>5s}: Top-1={res['top1']:.2%} | mAP={res['map']:.4f} (Số lượng={int(res['count'])})")

---
## Phần 7: Cẩm Nang Các Câu Lệnh Thực Thi (CLI Commands Guide)

Dưới đây là toàn bộ các câu lệnh dòng lệnh chuẩn đã được bản địa hóa và cấu hình đầy đủ bằng tiếng Việt:

### 1. Lệnh Chạy Bộ Kiểm Thử Tự Động (Unit Tests Verification)
Chạy bộ kiểm thử để xác minh toàn bộ 15 bài kiểm tra toán học và tính hợp lệ của các module:
```powershell
# Chạy toàn bộ 15 unit tests & integration tests
python -m pytest imp/kd-moe-ced/tests -v

# Chạy riêng kiểm thử từng module cụ thể:
python -m pytest imp/kd-moe-ced/tests/test_frontend.py -v   # Kiểm thử chuẩn hóa RMS & 6 đặc trưng vật lý
python -m pytest imp/kd-moe-ced/tests/test_router.py -v     # Kiểm thử bộ định tuyến Router & cổng mềm alpha
python -m pytest imp/kd-moe-ced/tests/test_split_moe.py -v  # Kiểm thử Warm-start Identity & SplitMoEFFN
python -m pytest imp/kd-moe-ced/tests/test_end_to_end.py -v # Kiểm thử tích hợp toàn diện từ sóng âm đến logits
```

### 2. Lệnh Chạy Huấn Luyện (Training Commands)
```powershell
# Huấn luyện mặc định 15 epoch với batch size 64
python imp/kd-moe-ced/train_kd_moe.py --epochs 15 --batch-size 64

# Huấn luyện tùy chỉnh tốc độ học phân tầng cho từng thành phần
python imp/kd-moe-ced/train_kd_moe.py \
    --epochs 20 \
    --batch-size 64 \
    --lr-backbone 4.0e-5 \
    --lr-experts 1.2e-4 \
    --lr-router 2.0e-4 \
    --data-root /marimo/datasets/mix-dataset-v2 \
    --cache-dir /marimo/datasets \
    --out-dir imp/kd-moe-ced/checkpoints
```

### 3. Lệnh Chạy Đánh Giá & Suy Luận (Inference & Evaluation Commands)
```powershell
# 1. Chạy suy luận độc lập mô hình học sinh (Standalone Student, 100% Zero-Oracle, ~6.62M tham số):
python imp/kd-moe-ced/champion_pipeline.py --mode standalone

# 2. Chạy suy luận kết hợp Ensemble động với hai giáo viên BEATs:
python imp/kd-moe-ced/champion_pipeline.py --mode self_contained

# 3. Chạy suy luận với batch size lớn (128) trên GPU:
python imp/kd-moe-ced/champion_pipeline.py \
    --mode standalone \
    --student-ckpt imp/kd-moe-ced/checkpoints/best_model.pt \
    --batch-size 128
```

---
### Kết Luận & Hướng Mở Rộng
Kiến trúc `CED-SingleMoE-SplitFFN` giải quyết triệt để vấn đề rò rỉ âm tiết tiếng nói từ bộ khử nhiễu **FRCRN** mà không cần tới nhãn Oracle SNR khi triển khai. Nhờ cấu trúc **Split-FFN** tại Blocks 10–11 và **Bộ định tuyến tự thân 22 chiều**, mô hình giữ được độ nhẹ xuất sắc (~6.62M tham số, dung lượng ~26.5 MB), sẵn sàng đóng gói và triển khai thời gian thực trên các thiết bị biên.